# B10_fractal. Generalization: held-out fractal type

Trained on fBm only, how does the model do on ridged terrain, against B9?

See `docs/plans/experiments-plan.md`, entry B10_fractal. Results are read from `artifacts/results/B10_fractal/`. Nothing here trains or samples.

## Run

Run these from the repo root, in order. `uv run erosion-sim list` shows what is already trained.

**1. Data.** Skip any that already exist.

```sh
sbatch slurm/datagen.sbatch main_256
```

**2. Train.** One command per model; each resumes if stopped.

```sh
uv run erosion-sim train B10_fractal fbm_only
```

- `final` is `B9/seed0`, trained in B9. If it is not trained yet, `uv run erosion-sim train B10_fractal final` trains it there.

**3. Evaluate.**

```sh
uv run erosion-sim evaluate B10_fractal
```

On the cluster, `uv run erosion-sim submit B10_fractal` does steps 2 and 3 as Slurm jobs.

Then run the cells below.

## How to read this

**Setup.** Each test pair is a terrain at a start time and the simulator's terrain some years later (`dT`).
Every method predicts the end terrain from the start, and is scored on every cell. The simulator follows the
stream power law with hillslope diffusion: `dz/dt = -K A^m |∇z| + D ∇²z`, where `K` is erodibility, `A`
drainage area (the upstream area draining through a cell), `m` the area exponent, and `D` the diffusion rate.

**Entries (table rows).**

- `no change`: the start terrain. The floor every method must beat.
- `analytical`: the stream power law solved exactly (Tzathas et al. 2024). Exact for channel cutting alone;
  misses hillslope smoothing and rivers changing course.
- each model by label; `<label>@<steps>` in a step sweep; `<label> + pit filling` after filling pits;
  `<label> chained <n>x<years>` for `n` calls in a row.

**Metrics.** Channels are the cells with the top 5% of the simulator's drainage area in that pair.

| Metric | What it is | Good |
|---|---|---|
| `rmse` | Root mean square height error (m) | Low |
| `skill` | `1 - MSE / MSE(no change)` over all pairs: share of the true change's error removed | 1 perfect, 0 = no change, negative = worse than doing nothing |
| `median_skill` | Median of per-pair skill. Less swayed by a few big pairs | As above |
| `erosion_ratio` | Mean absolute change over the simulator's | 1. Below is too little change, above too much |
| `iou` | Channel overlap: intersection over union of predicted and true channel cells | 1 perfect, 0 none |
| `iou_1` | Same, allowing one cell of slack | 1. Much higher than `iou` means channels are in the right place but slightly off |
| `log_area_rmse` | RMSE of log10 drainage area. 0.3 is about a factor of 2 | Low |
| `pits_per_km2` | Cells with no lower neighbor, per km². Pits trap water and break drainage | Low; compare with `no change` |
| `slope_area_error` | Gap between the method's and the simulator's slope-area exponent (how channel slope falls with drainage area) | Low |
| `fine_detail` | Change power at wavelengths of 4 cells or less, over the simulator's | 1. Below is too smooth, above is noise |
| `spread_correlation` | Generative models: correlation of per-cell spread across samples with per-cell error | High: the model is unsure where it is wrong |
| `seconds_per_pair` | Model time per pair | Low |

**Plots.**

- *Breakdowns* (`by_table`, `plot_by`): skill per group of one factor, one line per entry. Factors: `K_ref`
  (erodibility), `D_over_K` (diffusion relative to erodibility), `m`, `dT` (elapsed years), terrain noise
  setup, erodibility map shape and contrast, and the held-out parameter in out-of-distribution tests. A line
  dropping in one group shows where a method fails.
- *Spectrum* (`plot_spectrum`): change power at each wavelength over the simulator's. Fine detail is on the
  right. 1 is matching detail; below 1 is smoother, above is noisier.
- *Examples* (`plot_examples`): per row, the start terrain (shaded relief), then the height change of the
  simulator and each entry. Red is lowered (erosion), blue raised (deposition), on one shared scale per row.
  Compare channel positions and sharpness with the simulator's panel.
- *Step sweep* (`plot_steps`): metrics against network calls per sample. Look for where it stops improving.

**Training plot.** Left: train and validation loss; both should fall and level off. Validation rising while
train keeps falling is overfitting (the best-validation checkpoint is the one scored). Right: validation
skill, `1 - MSE / MSE(no change)`: 1 is perfect, 0 is no better than copying the input. For generative
models it is a one-step estimate, so read its trend, not its value.

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.experiments import registry
from erosion_sim.experiments.results import read_results
from erosion_sim.experiments.runner import model_folder
from erosion_sim.training.train import plot_history

NAME = "B10_fractal"
exp = registry.get(NAME)
results = read_results(NAME)
print(exp.question)
{"models": {label: m["folder"] for label, m in results.manifest["models"].items()},
 **{k: results.manifest[k] for k in ("code", "time", "slurm_job")}}

## Training

In [ ]:
for label in exp.models:
    folder = model_folder(NAME, label)[0]
    print(label, folder)
    plot_history(json.loads((folder / "history.json").read_text()))

## Every metric, over all test pairs

In [ ]:
plots.table(results)

## Breakdowns

Each factor with more than one group among the test pairs.

In [ ]:
for factor in results.scalars["by"]:
    plots.by_table(results, factor, "skill")
    print()

In [ ]:
for factor in results.scalars["by"]:
    plots.plot_by(results, factor, "skill")

## Detail and examples

In [ ]:
plots.plot_spectrum(results)
plots.plot_examples(results)

## Conclusion

_Written after reading the results above._